In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import os

In [2]:
BASE_DIR = "../data"

# skiprows=4 skips the World Bank metadata header; drop redundant/artifact columns
education_spend_df = pd.read_csv(
    f"{BASE_DIR}/education-spend/country-education-spend.csv",
    skiprows = 4
).drop(columns=["Indicator Name", "Indicator Code", "Country Code", "Unnamed: 70"])

gini_df = pd.read_csv(
    f"{BASE_DIR}/GINI-index/country-GINI-index.csv",
    skiprows = 4
).drop(columns=["Indicator Name", "Indicator Code", "Country Code", "Unnamed: 70"])

corruption_df = pd.read_csv(
    f"{BASE_DIR}/corruption_index/country-corruption-index.csv",
    skiprows = 4
).drop(columns=["Indicator Name", "Indicator Code", "Country Code", "Unnamed: 70"])

regulation_df = pd.read_csv(
    f"{BASE_DIR}/regulation/CPIA.csv",
    skiprows = 4
).drop(columns=["Indicator Name", "Indicator Code", "Country Code", "Unnamed: 70"])

# Unlike the other two sources, this file has every country row duplicated verbatim; drop the
# duplicates so corruption_df lines up 1:1 with gini_df/education_spend_df like the other two do
corruption_df = corruption_df.drop_duplicates().reset_index(drop=True)

# Nauru is listed under its Nauruan name ("Naoero") here but as "Nauru" in the other two
# datasets -- align it so it isn't silently dropped from the later merge on "Country Name"
corruption_df["Country Name"] = corruption_df["Country Name"].replace("Naoero", "Nauru")

# Same Nauru/"Naoero" naming quirk shows up in this source too
regulation_df["Country Name"] = regulation_df["Country Name"].replace("Naoero", "Nauru")

In [3]:
# Year columns are strings; keep only non-year columns and years >= 2000
cols_to_drop = [col for col in gini_df.columns if str(col).strip().isdigit() and int(col) < 2000]
gini_df = gini_df.drop(columns=cols_to_drop)
education_spend_df = education_spend_df.drop(columns=cols_to_drop)
corruption_df = corruption_df.drop(columns=cols_to_drop)
regulation_df = regulation_df.drop(columns=cols_to_drop)

In [4]:
gini_df.sort_values("Country Name")
education_spend_df.sort_values("Country Name")
corruption_df.sort_values("Country Name")
regulation_df.sort_values("Country Name")

gini_df = gini_df.replace(["", "nan", "None"], np.nan)
education_spend_df = education_spend_df.replace(["", "nan", "None"], np.nan)
corruption_df = corruption_df.replace(["", "nan", "None"], np.nan)
regulation_df = regulation_df.replace(["", "nan", "None"], np.nan)

gini_df.isna()
education_spend_df.isna()
corruption_df.isna()
regulation_df.isna()

common_countries = list()
common_countries.extend(list(gini_df["Country Name"]) + list(education_spend_df["Country Name"]) +list(corruption_df["Country Name"]) + list(regulation_df["Country Name"]))
# Counter(common_countries) creates a dictionary of the number of times an object shows up in the list
counts = Counter(common_countries)
duplicates_only = [x for x in common_countries if counts[x] == 4]
common_countries_unique = list(set(duplicates_only))

# Finds the difference between the common countries list and the metric for each iteration
difference_gini = np.setdiff1d(gini_df["Country Name"], common_countries_unique, assume_unique=False)
gini_df = gini_df[~gini_df["Country Name"].isin(difference_gini)]

difference_education = np.setdiff1d(education_spend_df["Country Name"], common_countries_unique,assume_unique=False)
education_spend_df = education_spend_df[~education_spend_df["Country Name"].isin(difference_education)]

difference_corruption = np.setdiff1d(corruption_df["Country Name"], common_countries_unique, assume_unique=False)
corruption_df = corruption_df[~corruption_df["Country Name"].isin(difference_corruption)]

difference_regulation = np.setdiff1d(regulation_df["Country Name"], common_countries_unique,assume_unique=False)
regulation_df = regulation_df[~regulation_df["Country Name"].isin(difference_regulation)]

# Cell 5's job: drop any country that's entirely NaN across every year in ANY of the four sources.
# Values are already normalized to real NaN above, so a plain .isna() check is enough here.
year_cols = [c for c in gini_df.columns if c != "Country Name"]

gini_has_data = set(gini_df.loc[~gini_df[year_cols].isna().all(axis=1), "Country Name"])
education_has_data = set(education_spend_df.loc[~education_spend_df[year_cols].isna().all(axis=1), "Country Name"])
corruption_has_data = set(corruption_df.loc[~corruption_df[year_cols].isna().all(axis=1), "Country Name"])
regulation_has_data = set(regulation_df.loc[~regulation_df[year_cols].isna().all(axis=1), "Country Name"])

countries_with_full_data = gini_has_data & education_has_data & corruption_has_data & regulation_has_data

gini_df = gini_df[gini_df["Country Name"].isin(countries_with_full_data)]
education_spend_df = education_spend_df[education_spend_df["Country Name"].isin(countries_with_full_data)]
corruption_df = corruption_df[corruption_df["Country Name"].isin(countries_with_full_data)]
regulation_df = regulation_df[regulation_df["Country Name"].isin(countries_with_full_data)]

gini_df

,Country Name,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
4,Angola,51.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.7,...,NaN,NaN,51.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Albania,NaN,NaN,31.7,NaN,NaN,30.6,NaN,NaN,30.0,...,33.7,33.1,30.1,30.1,29.4,NaN,NaN,NaN,NaN,NaN
10,Armenia,NaN,35.4,34.8,33.0,37.5,36.0,29.7,31.2,29.2,...,32.5,33.6,34.4,30.0,25.1,27.9,27.9,27.2,27.4,NaN
15,Azerbaijan,NaN,36.5,25.3,26.8,26.6,26.6,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,Burundi,NaN,NaN,NaN,NaN,NaN,NaN,33.4,NaN,NaN,...,NaN,NaN,NaN,NaN,37.5,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,Vanuatu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,32.3,NaN,NaN,NaN,NaN,NaN,NaN
259,Samoa,NaN,NaN,40.7,NaN,NaN,NaN,NaN,NaN,42.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
261,"Yemen, Rep.",NaN,NaN,NaN,NaN,NaN,34.7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
263,Zambia,NaN,NaN,42.1,NaN,54.3,NaN,54.6,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,51.5,NaN,NaN,NaN


In [5]:
year_cols_numeric = [c for c in gini_df.columns if c != "Country Name"]
gini_df[year_cols_numeric] = gini_df[year_cols_numeric].round(3)
education_spend_df[year_cols_numeric] = education_spend_df[year_cols_numeric].round(3)
corruption_df[year_cols_numeric] = corruption_df[year_cols_numeric].round(3)
regulation_df[year_cols_numeric] = regulation_df[year_cols_numeric].round(3)

education_spend_df

,Country Name,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
4,Angola,2.608,NaN,NaN,NaN,NaN,2.120,2.281,NaN,NaN,...,2.755,2.467,2.184,2.073,2.667,2.297,2.385,2.513,NaN,NaN
5,Albania,3.330,3.342,3.003,3.035,3.131,3.200,3.103,3.146,NaN,...,3.920,3.547,3.082,3.870,3.325,3.006,2.730,3.092,NaN,NaN
10,Armenia,2.773,2.469,2.135,2.145,2.487,2.712,2.716,3.019,3.173,...,2.758,2.708,2.113,2.160,2.706,2.768,2.492,2.439,NaN,NaN
15,Azerbaijan,3.854,3.503,3.154,3.286,3.448,2.975,2.556,2.549,2.441,...,2.903,2.474,2.455,3.176,4.332,3.702,3.047,3.660,NaN,NaN
16,Burundi,2.645,2.904,3.005,NaN,3.745,3.632,NaN,NaN,5.193,...,4.692,4.762,5.079,5.347,5.322,4.871,NaN,NaN,NaN,4.504
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,Vanuatu,7.004,8.951,8.132,8.442,NaN,NaN,NaN,NaN,5.952,...,7.677,4.534,8.606,1.774,2.174,10.110,10.703,7.581,7.637,NaN
259,Samoa,3.631,3.977,3.939,NaN,NaN,NaN,NaN,NaN,5.117,...,3.981,4.189,4.472,4.584,4.503,5.216,6.160,6.110,5.456,NaN
261,"Yemen, Rep.",9.662,9.245,NaN,NaN,NaN,NaN,NaN,NaN,5.151,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
263,Zambia,1.788,NaN,NaN,NaN,2.463,1.736,NaN,1.241,1.100,...,3.748,3.730,4.740,4.418,3.944,3.114,3.659,4.074,NaN,NaN


In [6]:
corruption_df

,Country Name,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
4,Angola,21.937,NaN,21.457,21.072,23.102,21.661,21.606,21.937,22.150,...,17.731,19.095,23.934,28.326,29.803,34.867,35.646,35.219,35.253,NaN
5,Albania,27.962,NaN,27.606,27.452,32.564,29.981,33.792,34.188,34.823,...,37.609,36.391,35.971,36.955,34.507,35.669,37.725,38.462,39.338,NaN
10,Armenia,31.163,NaN,28.827,28.799,34.249,35.735,35.127,33.572,32.479,...,36.101,36.436,44.530,46.795,49.992,48.985,49.019,49.972,50.783,NaN
15,Azerbaijan,19.891,NaN,19.683,22.253,22.642,24.423,24.738,24.251,24.075,...,32.627,30.476,29.506,32.571,32.475,32.151,31.050,30.003,32.166,NaN
16,Burundi,29.444,NaN,33.672,33.633,28.459,26.411,26.254,24.822,24.612,...,21.732,21.285,18.208,18.087,20.469,19.707,21.065,21.702,18.252,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,Vanuatu,50.756,NaN,50.516,50.433,46.072,54.160,54.240,56.192,56.206,...,45.467,46.511,44.083,44.107,47.314,47.126,45.107,44.104,44.806,NaN
259,Samoa,46.660,NaN,46.473,46.409,44.447,53.359,53.472,53.391,53.413,...,51.675,60.320,60.373,60.446,64.269,63.375,63.363,64.430,61.783,NaN
261,"Yemen, Rep.",29.471,NaN,26.192,25.313,24.272,26.061,31.513,30.912,32.334,...,15.264,15.651,12.371,12.600,15.474,15.629,14.127,15.426,14.742,NaN
263,Zambia,29.084,NaN,31.625,31.472,35.597,35.630,39.919,40.827,40.853,...,39.276,38.462,35.375,34.787,35.607,33.830,39.392,39.796,38.608,NaN


In [7]:
# how="right" keeps every row in gini_df even when a country has no education-spend match;
# suffixes disambiguate the duplicate year columns coming from each source
combined = pd.merge(education_spend_df, gini_df, on="Country Name", how="right", suffixes=("_edu", "_gini"))
combined = pd.merge(combined, corruption_df, on="Country Name", how="left")
combined = pd.merge(combined, regulation_df, on="Country Name", how="left", suffixes=("", "_reg"))

year_cols = [col for col in gini_df.columns if col != "Country Name"]

# Collapse each year's _gini/_edu/corruption/regulation columns into a single
# [gini, education_spend, corruption, regulation] quadruple; NaN if any value is missing,
# so downstream filtering only needs one check
for year in year_cols:
    quadruples = combined[[f"{year}_gini", f"{year}_edu", year, f"{year}_reg"]].values.tolist()
    combined[year] = [np.nan if any(pd.isna(v) for v in quadruple) else quadruple for quadruple in quadruples]
    combined = combined.drop(columns=[f"{year}_gini", f"{year}_edu", f"{year}_reg"])

combined

,Country Name,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Angola,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Albania,NaN,NaN,NaN,NaN,NaN,"[30.6, 3.2, 29.981, 3.5]",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Armenia,NaN,NaN,NaN,NaN,NaN,"[36.0, 2.712, 35.735, 4.0]","[29.7, 2.716, 35.127, 4.0]","[31.2, 3.019, 33.572, 4.0]","[29.2, 3.173, 32.479, 4.0]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Azerbaijan,NaN,NaN,NaN,NaN,NaN,"[26.6, 2.975, 24.423, 3.5]",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Burundi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"[37.5, 5.322, 20.469, 3.0]",NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,Vanuatu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,"[32.3, 1.774, 44.107, 3.0]",NaN,NaN,NaN,NaN,NaN,NaN
79,Samoa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[42.0, 5.117, 53.413, 3.5]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80,"Yemen, Rep.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
81,Zambia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,"[51.5, 3.659, 39.392, 4.0]",NaN,NaN,NaN


In [8]:
combined.to_csv("../data/processed/combined_cleaned.csv", index=False)
gini_df.to_csv("../data/processed/gini_cleaned.csv", index=False)
education_spend_df.to_csv("../data/processed/education_cleaned.csv", index=False)
corruption_df.to_csv("../data/processed/corruption_cleaned.csv", index=False)
regulation_df.to_csv("../data/processed/regulation_cleaned.csv", index=False)

In [9]:
# Reshape from wide (one column per year) to long (one row per country-year) format
melted = combined.melt(id_vars="Country Name", var_name="Year", value_name="Values")

# Keep only rows where a real [gini, edu, corruption, regulation] quadruple survived (drops the NaN placeholders)
melted = melted[melted["Values"].apply(lambda v: isinstance(v, list))].copy()

# NOTE: this overwrites the "Year" column melt() already set correctly from the original column
# names, renumbering each country's remaining rows sequentially from 2000 via cumcount(). That's
# only correct if a country's data has no gaps -- any missing year shifts every later year off by one.
melted["Year"] = melted.groupby("Country Name").cumcount() + 2000

melted["GINI"] = melted["Values"].apply(lambda v: v[0])
melted["Education_Spend"] = melted["Values"].apply(lambda v: v[1])
melted["Corruption"] = melted["Values"].apply(lambda v: v[2])
melted["Regulation"] = melted["Values"].apply(lambda v: v[3])

reshaped_GINI_education_df = melted[["Country Name", "Year", "GINI", "Education_Spend", "Corruption", "Regulation"]].rename(
    columns={"Country Name": "Country"}
).reset_index(drop=True)

reshaped_GINI_education_df.sort_values("Country")

,Country,Year,GINI,Education_Spend,Corruption,Regulation
0,Albania,2000,30.6,3.200,29.981,3.5
1,Armenia,2000,36.0,2.712,35.735,4.0
52,Armenia,2005,30.0,3.249,34.265,4.0
19,Armenia,2002,31.2,3.019,33.572,4.0
99,Armenia,2008,30.6,2.650,37.043,4.5
...,...,...,...,...,...,...
98,Viet Nam,2002,35.6,4.408,38.492,3.5
125,Viet Nam,2003,34.8,3.540,40.529,3.5
67,Zambia,2000,52.0,3.416,40.385,3.5
222,Zambia,2002,51.5,3.659,39.392,4.0


In [10]:
reshaped_GINI_education_df.to_csv("../data/processed/long_form.csv")